# Prepare 60K Demo Dataset

Creates a **60,000-sample balanced demo dataset** (30K benign + 30K attack) from `checkpoint_balanced_full.parquet`.

Outputs saved to Google Drive:
- `X_test_demo_60k.csv` — normalized feature matrix
- `y_test_demo_60k.csv` — labels (0=benign, 1=attack)

> **Note:** The test attack partition has ~143K samples, so 30K attack is safely within bounds.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import os, json
import numpy as np
import pandas as pd

# -------------------------------------------------------
# 1. LOCATE PARQUET
# -------------------------------------------------------
parquet_candidates = [
    '/content/drive/MyDrive/IDS_Dashboard_Submission/models/checkpoint_balanced_full.parquet',
    '/content/drive/MyDrive/test/checkpoint_balanced_full.parquet',
    '/content/drive/MyDrive/checkpoint_balanced_full.parquet',
]
PARQUET_FILE = next((p for p in parquet_candidates if os.path.exists(p)), None)
if PARQUET_FILE is None:
    raise FileNotFoundError('checkpoint_balanced_full.parquet not found. Check Drive mount.')
print(f'Parquet: {PARQUET_FILE}')

# -------------------------------------------------------
# 2. LOCATE BOUNDS FILE
# -------------------------------------------------------
bounds_candidates = [
    '/content/drive/MyDrive/IDS_Dashboard_Submission/models/X_bounds_cic.json',
    '/content/drive/MyDrive/test/X_bounds_cic.json',
    '/content/drive/MyDrive/X_bounds_cic.json',
]
BOUNDS_FILE = next((p for p in bounds_candidates if os.path.exists(p)), None)
if BOUNDS_FILE is None:
    raise FileNotFoundError('X_bounds_cic.json not found.')
print(f'Bounds: {BOUNDS_FILE}')

Parquet: /content/drive/MyDrive/IDS_Dashboard_Submission/models/checkpoint_balanced_full.parquet
Bounds: /content/drive/MyDrive/IDS_Dashboard_Submission/models/X_bounds_cic.json


In [10]:
# -------------------------------------------------------
# 3. LOAD PARQUET
# -------------------------------------------------------
print('Loading parquet (this may take a minute)...')
df = pd.read_parquet(PARQUET_FILE)
print(f'Total rows: {len(df):,}')

for col in df.columns:
    df[col] = df[col].astype(np.int32 if col == 'Label' else np.float32)

feature_cols = df.columns.drop('Label').tolist()
print(f'Features: {len(feature_cols)} | Benign: {(df["Label"]==0).sum():,} | Attack: {(df["Label"]==1).sum():,}')

Loading parquet (this may take a minute)...
Total rows: 5,493,868
Features: 77 | Benign: 2,746,934 | Attack: 2,746,934


In [11]:
# -------------------------------------------------------
# 4. SAMPLE 30K BENIGN + 30K ATTACK
#    Same partition offsets as prepare_demo_data.py
#    Training consumed [0:1576063], test starts at [1576063:]
# -------------------------------------------------------
SAMPLE_PER_CLASS = 30_000

benign_idx = df[df['Label'] == 0].index.values.copy()
attack_idx = df[df['Label'] == 1].index.values.copy()

np.random.seed(42)
np.random.shuffle(benign_idx)
np.random.shuffle(attack_idx)

test_benign_idx = benign_idx[1576063 : 1576063 + 1170871]
test_attack_idx = attack_idx[1576063 : 1576063 + 143849]

print(f'Test benign pool: {len(test_benign_idx):,} | Test attack pool: {len(test_attack_idx):,}')

test_benign_sample = df.loc[test_benign_idx].head(SAMPLE_PER_CLASS)
test_attack_sample = df.loc[test_attack_idx].head(SAMPLE_PER_CLASS)

print(f'Sampled: {len(test_benign_sample):,} benign + {len(test_attack_sample):,} attack')

demo_df = pd.concat([test_benign_sample, test_attack_sample]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Combined + shuffled: {len(demo_df):,} rows')

KeyboardInterrupt: 

In [ ]:
# -------------------------------------------------------
# 5. RECONSTRUCT NORMALISATION SCALER FROM BOUNDS
#    Identical method to prepare_demo_data.py
# -------------------------------------------------------
print(f'Reconstructing scaler from: {BOUNDS_FILE}')
with open(BOUNDS_FILE, 'r') as f:
    bounds = json.load(f)

means, stds = [], []
for col in feature_cols:
    raw_min = float(df[col].min())
    raw_max = float(df[col].max())
    norm_min = bounds[col]['min']
    norm_max = bounds[col]['max']
    if abs(norm_max - norm_min) < 1e-6:
        std, mean = 1e-6, raw_min
    else:
        std  = (raw_max - raw_min) / (norm_max - norm_min)
        mean = raw_max - norm_max * std
        if std < 0:
            std  = abs(std)
            mean = raw_max - norm_max * std
    means.append(mean)
    stds.append(std)

means = np.array(means, dtype=np.float32)
stds  = np.array(stds,  dtype=np.float32)
print('Scaler ready.')

Reconstructing scaler from: /content/drive/MyDrive/IDS_Dashboard_Submission/models/X_bounds_cic.json
Scaler ready.


In [ ]:
# -------------------------------------------------------
# 6. NORMALISE
# -------------------------------------------------------
X_raw  = demo_df[feature_cols].values
y_demo = demo_df['Label'].values

X_norm = (X_raw - means) / stds

X_demo_df = pd.DataFrame(X_norm, columns=feature_cols)
y_demo_df = pd.DataFrame(y_demo, columns=['Label'])

print(f'X: {X_demo_df.shape} | Benign: {(y_demo==0).sum():,} | Attack: {(y_demo==1).sum():,}')

X: (60000, 77) | Benign: 30,000 | Attack: 30,000


In [ ]:
# -------------------------------------------------------
# 7. SAVE TO GOOGLE DRIVE (primary) + KERNEL (backup)
# -------------------------------------------------------
gdrive_candidates = [
    '/content/drive/MyDrive/IDS_Dashboard_Submission/datasets/demo',
    '/content/drive/MyDrive/IDS_Dashboard_Submission/datasets',
    '/content/drive/MyDrive/test',
    '/content/drive/MyDrive',
]
GDRIVE_OUT = next((p for p in gdrive_candidates if os.path.exists(p)), '/content/drive/MyDrive')
os.makedirs(GDRIVE_OUT, exist_ok=True)

gdrive_x = os.path.join(GDRIVE_OUT, 'X_test_demo_60k.csv')
gdrive_y = os.path.join(GDRIVE_OUT, 'y_test_demo_60k.csv')

print('Saving to Google Drive...')
X_demo_df.to_csv(gdrive_x, index=False)
y_demo_df.to_csv(gdrive_y, index=False)
print(f'  X -> {gdrive_x}')
print(f'  y -> {gdrive_y}')

# Kernel backup
kernel_out = '/content/demo'
os.makedirs(kernel_out, exist_ok=True)
X_demo_df.to_csv(os.path.join(kernel_out, 'X_test_demo_60k.csv'), index=False)
y_demo_df.to_csv(os.path.join(kernel_out, 'y_test_demo_60k.csv'), index=False)
print(f'\nAlso saved to kernel: {kernel_out}')
print(f'\nDone! {len(X_demo_df):,} samples (30K benign + 30K attack).')

Saving to Google Drive...
  X -> /content/drive/MyDrive/IDS_Dashboard_Submission/datasets/demo/X_test_demo_60k.csv
  y -> /content/drive/MyDrive/IDS_Dashboard_Submission/datasets/demo/y_test_demo_60k.csv

Also saved to kernel: /content/demo

Done! 60,000 samples (30K benign + 30K attack).
